# 05 - Wave Equation

Solve $u_{tt} = u_{xx}$

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '../src')
from pinns import MLP, set_seed
from pinns.utils.derivatives import gradient
set_seed(42)
model = MLP(input_dim=2, output_dim=1, hidden_dims=[32,32,32], activation='tanh')

In [ ]:
def pde_loss(model, x, t):
    xt = torch.cat([x, t], dim=1).requires_grad_(True)
    u = model(xt)
    g = gradient(u, xt)
    u_x, u_t = g[:, 0:1], g[:, 1:2]
    u_xx = gradient(u_x, xt)[:, 0:1]
    u_tt = gradient(u_t, xt)[:, 1:2]
    return torch.mean((u_tt - u_xx)**2)
def bc_loss(model, n=50):
    t = torch.rand(n, 1)
    return torch.mean(model(torch.cat([torch.zeros(n,1), t], dim=1))**2 + model(torch.cat([torch.ones(n,1), t], dim=1))**2)
def ic_loss(model, n=50):
    x = torch.rand(n, 1)
    xt = torch.cat([x, torch.zeros(n, 1)], dim=1).requires_grad_(True)
    u0 = model(xt)
    ut0 = gradient(u0, xt)[:, 1:2]
    return torch.mean((u0 - torch.sin(np.pi*x))**2) + torch.mean(ut0**2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(800):
    opt.zero_grad()
    loss = pde_loss(model, torch.rand(50,1), torch.rand(50,1)) + 10*bc_loss(model) + 10*ic_loss(model)
    loss.backward(); opt.step()
    if (epoch+1) % 200 == 0: print(f'Epoch {epoch+1}: {loss.item():.4e}')

In [ ]:
x = torch.linspace(0, 1, 50).reshape(-1, 1)
with torch.no_grad(): u_pred = model(torch.cat([x, torch.full_like(x, 0.5)], dim=1))
u_exact = torch.sin(np.pi*x) * torch.cos(torch.tensor(np.pi*0.5))
plt.figure(figsize=(8,4))
plt.plot(x.numpy(), u_exact.numpy(), 'b-', lw=2, label='Exact')
plt.plot(x.numpy(), u_pred.numpy(), 'r--', lw=2, label='PINN')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xlabel('x'); plt.ylabel('u'); plt.title('Wave Equation t=0.5')
plt.tight_layout(); plt.show()
print(f'MAE: {torch.abs(u_pred - u_exact).mean():.6e}')